# A2 — Delegating

**Manager-worker delegation loop with a hard budget.** The manager LLM decides dynamically which workers to spawn.

**Fictional task**: draft a three-item FAQ about a fictional *urban beekeeping co-op*.

In [ ]:
# --- Load API key from the canonical env file (see memory `reference_api_keys`) ---
import os
from pathlib import Path

env_file = Path('/home/shumway/projects/meta-agents/.env')
if env_file.exists() and not os.environ.get('OPENROUTER_API_KEY'):
    for raw in env_file.read_text().splitlines():
        s = raw.strip()
        if s.startswith('OPENROUTER_API_KEY='):
            os.environ['OPENROUTER_API_KEY'] = s.split('=', 1)[1].strip().strip('"').strip("'")
            break

assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing'
# Default worker model for agents that do not declare their own (DAG engine consults LLM_MODEL).
os.environ.setdefault('LLM_MODEL', 'deepseek/deepseek-chat-v3.1')
print('OpenRouter key loaded. Default model:', os.environ['LLM_MODEL'])

## Load + A2 compliance

In [ ]:
from pathlib import Path
from awp.parser import parse_manifest, parse_agent
from awp.validator import check_compliance, AutonomyLevel

WORKFLOW_DIR = Path('/home/shumway/projects/agent-workflow-protocol/examples/workflows/08-delegation-loop')
manifest = parse_manifest(WORKFLOW_DIR / 'workflow.awp.yaml')
agents = {}
for ad in (WORKFLOW_DIR / 'agents').iterdir():
    a = ad / 'agent.awp.yaml'
    if a.exists():
        agents[ad.name] = parse_agent(a)

result = check_compliance(manifest, agents, target_level=AutonomyLevel.A2_DELEGATING)
assert result.level >= AutonomyLevel.A2_DELEGATING, f'Not A2: {result.errors}'
budget = manifest.orchestration.delegation_loop.budget
print(f'A2 compliant. Budget: max_loops={budget.max_loops}, max_workers={budget.max_total_workers}, max_tokens={budget.max_total_tokens}')

## Run the delegation loop

In [ ]:
import json
import logging
logging.basicConfig(level=logging.WARNING)

from awp.runtime import WorkflowRunner

TASK = (
    'Draft a concise FAQ with exactly three entries about running a small '
    'urban beekeeping co-op on a fictional rooftop. Each entry must have a question and a 2-3 sentence answer.'
)
runner = WorkflowRunner(
    WORKFLOW_DIR,
    manager_model='openai/gpt-5-mini',
    worker_model='deepseek/deepseek-chat-v3.1',
)
result = runner.run(TASK)

print(json.dumps({k: v for k, v in result.items() if not k.startswith('_')}, indent=2, default=str)[:3000])

## Assertions (E2E rubric)

In [ ]:
assert isinstance(result, dict) and result, 'empty result'
# Delegation loop returns a final state dict — look for non-empty content
has_content = any(
    (isinstance(v, (str, dict, list)) and bool(v)) for k, v in result.items() if not k.startswith('_')
)
assert has_content, f'no content in result. Keys: {list(result)}'
# If an agent output is present it must carry confidence (R17)
for k, v in result.items():
    if isinstance(v, dict) and 'confidence' in v:
        assert not v.get('error'), f'{k} failed: {v["error"]}'
print('OK — delegation loop closed with non-empty result.')